In [0]:
%sql
CREATE CATALOG IF NOT EXISTS ATMOS
  MANAGED LOCATION 'abfss://atmos-landing-dev-001v2@saatmosdevwestus2001v2.dfs.core.windows.net/';

CREATE SCHEMA   IF NOT EXISTS ATMOS.GOLD;

### 2. Criação/substituição da tabela Gold
#### Agregação Diária
O INMET fornece dados **horários** — agrega para diário antes de calcular médias mensais,
garantindo paridade com o Atmos (que já é diário e chega com `hora_observacao = null`).
#### Agregação Mensal
Climatograma por estação × ano × mês.
| Coluna | Descrição |
|---|---|
| `temperatura_media_c` | Média das temperaturas médias diárias |
| `temperatura_maxima_media_c` | Média das máximas diárias (normal climatológica de máximas) |
| `temperatura_minima_media_c` | Média das mínimas diárias (normal climatológica de mínimas) |
| `temperatura_maxima_abs_c` | Maior temperatura registrada no mês |
| `temperatura_minima_abs_c` | Menor temperatura registrada no mês |
| `precipitacao_total_mm` | Soma da precipitação diária do mês |
| `umidade_media_pct` | Média da umidade relativa diária |
| `pressao_media_hpa` | Média da pressão atmosférica diária |
| `amplitude_termica_media_c` | Amplitude térmica média diária (max - min) |
| `dias_com_dados` | Quantidade de dias com registros no mês |

In [0]:
%sql
CREATE OR REPLACE TABLE atmos.gold.perfil_sazonal_mensal
AS
WITH diario AS (
    SELECT
        data_observacao,
        id_estacao,
        cidade,
        AVG(temperatura_c)      AS temp_media_dia,
        MAX(temperatura_max_c)  AS temp_max_dia,
        MIN(temperatura_min_c)  AS temp_min_dia,
        SUM(precipitacao_mm)    AS precipitacao_dia_mm,
        AVG(umidade_pct)        AS umidade_media_dia,
        AVG(pressao_hpa)        AS pressao_media_dia_hpa
    FROM atmos.silver.climate_unified
    GROUP BY
        data_observacao,
        id_estacao,
        cidade
),

mensal AS (
    SELECT
        id_estacao,
        cidade,
        YEAR(data_observacao)                                       AS ano,
        MONTH(data_observacao)                                      AS mes,
        ROUND(AVG(temp_media_dia),                              2)  AS temperatura_media_c,
        ROUND(AVG(temp_max_dia),                                2)  AS temperatura_maxima_media_c,
        ROUND(AVG(temp_min_dia),                                2)  AS temperatura_minima_media_c,
        ROUND(MAX(temp_max_dia),                                2)  AS temperatura_maxima_abs_c,
        ROUND(MIN(temp_min_dia),                                2)  AS temperatura_minima_abs_c,
        ROUND(SUM(precipitacao_dia_mm),                         2)  AS precipitacao_total_mm,
        ROUND(AVG(umidade_media_dia),                           2)  AS umidade_media_pct,
        ROUND(AVG(pressao_media_dia_hpa),                       2)  AS pressao_media_hpa,
        ROUND(AVG(temp_max_dia - temp_min_dia),                 2)  AS amplitude_termica_media_c,
        COUNT(DISTINCT data_observacao)                             AS dias_com_dados,
        from_utc_timestamp(CURRENT_TIMESTAMP(), 'America/Sao_Paulo') AS timestamp_processamento
    FROM diario
    GROUP BY
        id_estacao,
        cidade,
        YEAR(data_observacao),
        MONTH(data_observacao)
)
SELECT * FROM mensal